# Dataset Augmentation Generator

Generates dataset variants for the Single Model architecture to test robustness across different conditions.

### Generated Variants:
- **D2a**: CLAHE + Mild Brightness/Contrast (Test impact of illumination normalization)
- **D2b**: D2a + Color Jitter (minimal) (Test model sensitivity to color variation)
- **D2c (full)**: D2b + Geometric (Flip/Rotation) (Test orientation invariance)

In [2]:
!pip install albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 582.8/582.8 kB 16.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 46.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [albumentations]


In [3]:
import os
import cv2
import ast
import shutil
from pathlib import Path
import numpy as np
import albumentations as A
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## Configuration
Set the input directory of the original YOLO dataset and the output base directory.

In [4]:
INPUT_DATASET_DIR = "/home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO"
OUTPUT_BASE_DIR = "/home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_Variants"
DATASET_FORMAT = "YOLO" # Support "YOLO" or "COCO"

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

ensure_dir(OUTPUT_BASE_DIR)


## Albumentations Pipelines

In [5]:
# Variant D2a: CLAHE + Mild Brightness/Contrast
pipline_d2a = A.Compose([
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0)
], keypoint_params=A.KeypointParams(format='xy'))

# Variant D2b: D2a + Color Jitter (minimal)
pipline_d2b = A.Compose([
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
    A.ColorJitter(brightness=0, contrast=0, saturation=0.1, hue=0.05, p=1.0)
], keypoint_params=A.KeypointParams(format='xy'))

# Variant D2c: D2b + Geometric (Flip/Rotation)
pipline_d2c = A.Compose([
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
    A.ColorJitter(brightness=0, contrast=0, saturation=0.1, hue=0.05, p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.SafeRotate(limit=45, border_mode=cv2.BORDER_CONSTANT, value=0, p=0.7)
], keypoint_params=A.KeypointParams(format='xy'))

PIPELINES = {
    'D2a': pipline_d2a,
    'D2b': pipline_d2b,
    'D2c': pipline_d2c
}

## Processing Logic
Read YOLO format instance segmentation annotations, handle pixel conversion for Albumentations keypoints, and write back normalized coordinates.

In [6]:
def process_image_and_label(img_path, label_path, out_img_path, out_label_path, transform, dataset_format="YOLO"):
    """
    Reads an image and its corresponding segmentation label,
    applies the transform, and saves the augmented outputs.
    Supports YOLO and standard COCO segmentation polygons.
    """
    image = cv2.imread(str(img_path))
    if image is None: return False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    classes = []
    keypoints = []
    keypoint_group_sizes = []

    if dataset_format == "YOLO":
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        class_id = parts[0]
                        pts = [float(p) for p in parts[1:]]
                        pts_xy = [(pts[i] * w, pts[i+1] * h) for i in range(0, len(pts)-1, 2)]
                        classes.append(class_id)
                        keypoints.extend(pts_xy)
                        keypoint_group_sizes.append(len(pts_xy))
    
    elif dataset_format == "COCO":
        # Note: True COCO implementations batch load the json at dataset level.
        # For simplicity in image-level iteration, logic assumes single annotation dicts.
        if os.path.exists(label_path):
            import json
            with open(label_path, 'r') as f:
                coco_ann = json.load(f)
                for ann in coco_ann.get('annotations', []):
                    class_id = str(ann['category_id'])
                    seg = ann['segmentation'][0]
                    pts_xy = [(seg[i], seg[i+1]) for i in range(0, len(seg)-1, 2)]
                    classes.append(class_id)
                    keypoints.extend(pts_xy)
                    keypoint_group_sizes.append(len(pts_xy))

    if len(keypoints) > 0:
        try:
            transformed = transform(image=image, keypoints=keypoints)
            trans_image, trans_keypoints = transformed['image'], transformed['keypoints']
        except Exception as e:
            print(f"Transform failed {img_path}: {e}")
            trans_image, trans_keypoints = image, keypoints
    else:
        transformed = transform(image=image, keypoints=[])
        trans_image, trans_keypoints = transformed['image'], []

    trans_image_bgr = cv2.cvtColor(trans_image, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(out_img_path), trans_image_bgr)

    if len(trans_keypoints) > 0:
        new_h, new_w = trans_image.shape[:2]
        
        if dataset_format == "YOLO":
            with open(out_label_path, 'w') as f:
                current_idx = 0
                for class_id, size in zip(classes, keypoint_group_sizes):
                    poly_pts = trans_keypoints[current_idx:current_idx+size]
                    current_idx += size
                    norm_pts = []
                    for (px, py) in poly_pts:
                        norm_pts.extend([f"{max(0.0, min(1.0, px/new_w)):.6f}", f"{max(0.0, min(1.0, py/new_h)):.6f}"])
                    if norm_pts:
                        f.write(f"{class_id} " + " ".join(norm_pts) + "\n")
                        
        elif dataset_format == "COCO":
            out_coco = {"annotations": []}
            current_idx = 0
            for class_id, size in zip(classes, keypoint_group_sizes):
                poly_pts = trans_keypoints[current_idx:current_idx+size]
                current_idx += size
                seg_flat = [float(coord) for point in poly_pts for coord in point]
                if seg_flat:
                    out_coco["annotations"].append({
                        "category_id": int(class_id),
                        "segmentation": [seg_flat]
                    })
            with open(out_label_path, 'w') as f:
                json.dump(out_coco, f)

    return True


## Generation Pipeline

In [7]:
def generate_variants():
    input_path = Path(INPUT_DATASET_DIR)
    
    if not input_path.exists():
        print(f"Warning: Dataset path {INPUT_DATASET_DIR} does not exist. Update the INPUT_DATASET_DIR variable.")
        return

    splits = ['train', 'valid', 'test']
    
    for variant_name, pipeline in PIPELINES.items():
        print(f"\n==== Generating Variant: {variant_name} ====")
        variant_dir = Path(OUTPUT_BASE_DIR) / f"{input_path.name}_{variant_name}"
        
        # Copy data.yaml if exists
        yaml_path = input_path / "data.yaml"
        if yaml_path.exists():
            ensure_dir(variant_dir)
            shutil.copy(yaml_path, variant_dir / "data.yaml")

        for split in splits:
            img_in_dir = input_path / split / "images"
            lbl_in_dir = input_path / split / "labels"
            
            if not img_in_dir.exists():
                continue
                
            img_out_dir = variant_dir / split / "images"
            lbl_out_dir = variant_dir / split / "labels"
            ensure_dir(img_out_dir)
            ensure_dir(lbl_out_dir)

            images = list(img_in_dir.glob("*.png")) + list(img_in_dir.glob("*.jpg"))
            
            print(f"Processing {split} split...")
            for img_path in tqdm(images):
                label_path = lbl_in_dir / (img_path.stem + ".txt")
                out_img = img_out_dir / img_path.name
                out_label = lbl_out_dir / (img_path.stem + ".txt")
                
                if split == 'train':
                    # Apply augmentations ONLY to the training set
                    process_image_and_label(img_path, label_path, out_img, out_label, pipeline, dataset_format=DATASET_FORMAT)
                else:
                    # For validation and test sets, copy unaltered images and labels to avoid data leakage
                    shutil.copy(img_path, out_img)
                    if label_path.exists():
                        shutil.copy(label_path, out_label)

    print("\n✅ Dataset variant generation completed.")

if __name__ == "__main__":
    generate_variants()


==== Generating Variant: D2a ====
Processing train split...


  0%|          | 0/123 [00:00<?, ?it/s]

Transform failed /home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO/train/images/LIMESTONE_DENSE_0005.png: Expected y for keypoint [613.6499 960.       0.       0.       0.    ] to be in range [0, 960), got 960.0
Expected y for keypoint [555.84 960.     0.     0.     0.  ] to be in range [0, 960), got 960.0
Expected y for keypoint [469.48993 960.        0.        0.        0.     ] to be in range [0, 960), got 960.0
Expected y for keypoint [538.2605 960.       0.       0.       0.    ] to be in range [0, 960), got 960.0
Expected y for keypoint [234.26944 960.        0.        0.        0.     ] to be in range [0, 960), got 960.0
Expected x for keypoint [1280.      122.5104    0.        0.        0.    ] to be in range [0, 1280), got 1280.0
Expected y for keypoint [932.2598 960.       0.       0.       0.    ] to be in range [0, 960), got 960.0
Expected y for keypoint [864.28033 960.        0.        0.        0.     ] to be in range [0, 960), got 960.0
Expected y for keypoint

  2%|▏         | 3/123 [00:00<00:07, 15.42it/s]

Transform failed /home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO/train/images/0007.png: Expected y for keypoint [464.57983 960.        0.        0.        0.     ] to be in range [0, 960), got 960.0
Expected y for keypoint [438.01984 960.        0.        0.        0.     ] to be in range [0, 960), got 960.0
Expected y for keypoint [767.49054 960.        0.        0.        0.     ] to be in range [0, 960), got 960.0
Expected x for keypoint [1280.      794.8896    0.        0.        0.    ] to be in range [0, 1280), got 1280.0
Expected x for keypoint [1280.      900.4896    0.        0.        0.    ] to be in range [0, 1280), got 1280.0
Expected x for keypoint [1280.       309.86017    0.         0.         0.     ] to be in range [0, 1280), got 1280.0


...

Transform failed /home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO/train/images/PIC-0018.jpg: Expected x for keypoint [10368.      3743.0017     0.         0.         0.    ] to be in range [0, 10368), got 10368.0
Expected x for keypoint [10368.      1572.0018     0.         0.         0.    ] to be in range [0, 10368), got 10368.0
Expected y for keypoint [2532.0005 5888.        0.        0.        0.    ] to be in range [0, 5888), got 5888.0
Expected y for keypoint [3228.9995 5888.        0.        0.        0.    ] to be in range [0, 5888), got 5888.0
Expected y for keypoint [6231.002 5888.       0.       0.       0.   ] to be in range [0, 5888), got 5888.0
Expected y for keypoint [8401.004 5888.       0.       0.       0.   ] to be in range [0, 5888), got 5888.0
Expected x for keypoint [10368.      4572.9976     0.         0.         0.    ] to be in range [0, 10368), got 10368.0
Expected x for keypoint [10368.     3818.998     0.        0.        0.   ] to be in range [0

 72%|███████▏  | 88/123 [00:06<00:03, 10.70it/s]

Transform failed /home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO/train/images/WIN_20260302_11_36_40_Pro.jpg: Expected x for keypoint [3840.     1410.0005    0.        0.        0.    ] to be in range [0, 3840), got 3840.0
Expected x for keypoint [3840.      961.9992    0.        0.        0.    ] to be in range [0, 3840), got 3840.0
Expected x for keypoint [3840.     1633.9989    0.        0.        0.    ] to be in range [0, 3840), got 3840.0
Expected x for keypoint [3840.     1419.9991    0.        0.        0.    ] to be in range [0, 3840), got 3840.0
Expected y for keypoint [1346.0006 2160.        0.        0.        0.    ] to be in range [0, 2160), got 2160.0
Expected y for keypoint [1439.0016 2160.        0.        0.        0.    ] to be in range [0, 2160), got 2160.0
Expected y for keypoint [1171.0004 2160.        0.        0.        0.    ] to be in range [0, 2160), got 2160.0
Expected y for keypoint [1332.9984 2160.        0.        0.        0.    ] to be in ra

 72%|███████▏  | 88/123 [00:06<00:02, 13.90it/s]


KeyboardInterrupt: 